# Clase 058 — Optuna: HPO bayesiano dedicado

TPE sampler + MedianPruner sobre GradientBoostingClassifier. Comparamos vs GridSearchCV con presupuesto similar.

Instalar: `pip install optuna`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.metrics import roc_auc_score
import time

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    OPTUNA_OK = True
except ImportError:
    print('optuna no instalado: `pip install optuna`')
    OPTUNA_OK = False

np.random.seed(42)

## 1. Dataset sintético

In [ ]:
X, y = make_classification(n_samples=2000, n_features=20, n_informative=10,
                            n_redundant=5, weights=[0.7, 0.3], random_state=42)
print('X', X.shape, 'pos rate', y.mean().round(3))
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

## 🧠 Intuición previa

Buscar hiperparámetros con grilla o al azar es como buscar las llaves probando cajones sin memoria: cada intento ignora lo aprendido en los anteriores. **Optuna con TPE** es una búsqueda *bayesiana*: mira los resultados que ya obtuvo y decide dónde mirar después, concentrando los intentos en las zonas que vienen dando buen score en vez de repartirlos a ciegas. El **pruner** agrega un atajo: si un trial arranca claramente peor que la mediana de los anteriores, lo mata antes de terminar y ahorra cómputo para los candidatos prometedores.

## 2. Objective + TPE sampler + MedianPruner

In [ ]:
def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 50, 300),
        'max_depth':         trial.suggest_int('max_depth', 2, 8),
        'learning_rate':     trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
    }
    aucs = []
    for step, (tr, te) in enumerate(cv.split(X, y)):
        model = GradientBoostingClassifier(random_state=42, **params)
        model.fit(X[tr], y[tr])
        auc = roc_auc_score(y[te], model.predict_proba(X[te])[:, 1])
        aucs.append(auc)
        trial.report(np.mean(aucs), step)
        if trial.should_prune():
            raise optuna.TrialPruned()
    return float(np.mean(aucs))

In [ ]:
if OPTUNA_OK:
    sampler = optuna.samplers.TPESampler(seed=42)
    pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=1)
    study = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)
    t0 = time.perf_counter()
    study.optimize(objective, n_trials=50, show_progress_bar=False)
    t_optuna = time.perf_counter() - t0
    print(f'Optuna best AUC: {study.best_value:.4f}')
    print(f'Optuna best params: {study.best_params}')
    print(f'completed {len([t for t in study.trials if t.state.name == "COMPLETE"])}/50, '
          f'pruned {len([t for t in study.trials if t.state.name == "PRUNED"])}, time {t_optuna:.1f}s')

## 3. GridSearchCV con presupuesto similar (~48 combos)

In [ ]:
param_grid = {
    'n_estimators':  [50, 100],
    'max_depth':     [3, 5],
    'learning_rate': [0.1],
    'subsample':     [1.0],
    'min_samples_split': [2, 10],
}
# 2 × 2 × 1 × 1 × 2 = 8 combos (grid reducido para que corra rapido en clase)

t0 = time.perf_counter()
gs = GridSearchCV(GradientBoostingClassifier(random_state=42),
                   param_grid, cv=cv, scoring='roc_auc', n_jobs=1)
gs.fit(X, y)
t_grid = time.perf_counter() - t0
print(f'Grid best AUC: {gs.best_score_:.4f}')
print(f'Grid best params: {gs.best_params_}')
print(f'8 combos, time {t_grid:.1f}s')

## 4. Comparativa

In [ ]:
if OPTUNA_OK:
    summary = pd.DataFrame({
        'estrategia': ['GridSearchCV (8)', 'Optuna TPE (50)'],
        'AUC':  [gs.best_score_, study.best_value],
        'time_s': [t_grid, t_optuna],
    }).round(4)
    print(summary.to_string(index=False))

## 5. Param importances (fANOVA, matplotlib)

In [ ]:
if OPTUNA_OK:
    importances = optuna.importance.get_param_importances(study)
    names = list(importances.keys())
    vals = list(importances.values())
    fig, ax = plt.subplots(figsize=(7, 3))
    ax.barh(names[::-1], vals[::-1], color='#37a')
    ax.set_xlabel('importancia fANOVA')
    ax.set_title('Importancia de hiperparámetros')
    plt.tight_layout()
    plt.show()

## 6. Historia de optimización

In [ ]:
if OPTUNA_OK:
    values = [t.value for t in study.trials if t.value is not None]
    best_so_far = np.maximum.accumulate(values)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.scatter(range(len(values)), values, alpha=0.5, label='trial AUC')
    ax.plot(range(len(values)), best_so_far, color='red', label='best so far')
    ax.set_xlabel('trial')
    ax.set_ylabel('AUC')
    ax.set_title('Optuna optimization history')
    ax.legend()
    plt.tight_layout()
    plt.show()

## Ejercicios

1. Cambiá a `CmaEsSampler` (puramente continuo). ¿Converge más rápido?
2. Implementá multi-objective (`directions=['maximize', 'minimize']`) optimizando AUC y tiempo de inferencia.
3. Persistí el study con `storage='sqlite:///opt.db'` y verificá `load_if_exists=True`.

## Conclusiones

- TPE explora el espacio de forma inteligente — alcanza mejor o igual AUC que Grid con menos trials efectivos.
- `MedianPruner` mata trials malos temprano — ahorra cómputo sin perder buenos candidatos.
- `plot_param_importances` orienta dónde poner esfuerzo — típicamente `learning_rate` y `max_depth` dominan en GBM.

## ✅ Soluciones de los ejercicios

Los 5 ejercicios del README usan Optuna. Cada solución intenta la versión con Optuna y, si no está instalado (no es una de las librerías base del curso), cae en un equivalente con scikit-learn / xgboost que ilustra el mismo concepto y siempre corre. Reutilizamos `X`, `y` y `cv` del notebook.

**Ej. 1 — Objective básico.** Tunear `LogisticRegression(C, penalty)` y `RandomForest(n_estimators, max_depth)` con TPE (50 trials). Fallback: random search manual.

In [ ]:

import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def obj_lr(t):
        m = LogisticRegression(C=t.suggest_float("C", 1e-3, 1e2, log=True),
                               penalty=t.suggest_categorical("penalty", ["l1", "l2"]),
                               solver="liblinear", max_iter=1000)
        return cross_val_score(m, X, y, cv=cv, scoring="roc_auc").mean()
    s_lr = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    s_lr.optimize(obj_lr, n_trials=50)
    def obj_rf(t):
        m = RandomForestClassifier(n_estimators=t.suggest_int("n_estimators", 50, 300),
                                   max_depth=t.suggest_int("max_depth", 2, 12),
                                   random_state=42, n_jobs=1)
        return cross_val_score(m, X, y, cv=cv, scoring="roc_auc").mean()
    s_rf = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    s_rf.optimize(obj_rf, n_trials=50)
    print(f"LogReg best AUC={s_lr.best_value:.4f} {s_lr.best_params}")
    print(f"RF     best AUC={s_rf.best_value:.4f} {s_rf.best_params}")
    assert s_lr.best_value > 0.5 and s_rf.best_value > 0.5
except ImportError:
    rng = np.random.default_rng(42)
    best = (0.0, None)
    for _ in range(20):
        C = float(10 ** rng.uniform(-3, 2)); pen = str(rng.choice(["l1", "l2"]))
        m = LogisticRegression(C=C, penalty=pen, solver="liblinear", max_iter=1000)
        auc = cross_val_score(m, X, y, cv=cv, scoring="roc_auc").mean()
        if auc > best[0]:
            best = (auc, {"C": round(C, 4), "penalty": pen})
    print("optuna no instalado -> random search manual (20 trials)")
    print(f"LogReg best AUC={best[0]:.4f} {best[1]}")
    assert best[0] > 0.5

**Ej. 2 — Search space compuesto (condicional).** `solver='liblinear'` admite `l1`/`l2`; otros solvers solo `l2`. Optuna maneja el condicional con un simple `if`. Fallback: enumeramos solo las combinaciones válidas.

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def obj(t):
        solver = t.suggest_categorical("solver", ["liblinear", "lbfgs"])
        if solver == "liblinear":
            penalty = t.suggest_categorical("penalty_lib", ["l1", "l2"])   # condicional
        else:
            penalty = "l2"                                                  # lbfgs solo l2
        m = LogisticRegression(C=t.suggest_float("C", 1e-2, 1e2, log=True),
                               solver=solver, penalty=penalty, max_iter=2000)
        return cross_val_score(m, X, y, cv=cv, scoring="roc_auc").mean()
    st = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
    st.optimize(obj, n_trials=40)
    print(f"best AUC={st.best_value:.4f} {st.best_params}")
    assert st.best_value > 0.5
except ImportError:
    combos = [("liblinear", "l1"), ("liblinear", "l2"), ("lbfgs", "l2")]  # solo validas
    best = max(((cross_val_score(LogisticRegression(C=1.0, solver=s, penalty=p, max_iter=2000),
                                 X, y, cv=cv, scoring="roc_auc").mean(), s, p) for s, p in combos))
    print("optuna no instalado -> enumeracion de combinaciones (solver, penalty) validas")
    print(f"best AUC={best[0]:.4f} solver={best[1]} penalty={best[2]}")
    print("clave: lbfgs NO admite l1; el espacio de busqueda es condicional")
    assert best[0] > 0.5

**Ej. 3 — Pruning en XGBoost.** Un pruner mata trials/iteraciones que no prometen. El concepto equivalente y disponible sin Optuna es el *early stopping*: XGBoost corta el boosting cuando la validación deja de mejorar.

In [ ]:

import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

Xt, Xv, yt, yv = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
try:
    import optuna
    from optuna.integration import XGBoostPruningCallback
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def obj(t):
        params = {"objective": "binary:logistic", "eval_metric": "auc",
                  "max_depth": t.suggest_int("max_depth", 2, 8),
                  "eta": t.suggest_float("eta", 1e-2, 0.3, log=True)}
        dtr, dv = xgb.DMatrix(Xt, yt), xgb.DMatrix(Xv, yv)
        cb = XGBoostPruningCallback(t, "validation-auc")
        bst = xgb.train(params, dtr, num_boost_round=300,
                        evals=[(dv, "validation")], callbacks=[cb], verbose_eval=False)
        return roc_auc_score(yv, bst.predict(dv))
    st = optuna.create_study(direction="maximize",
                             pruner=optuna.pruners.MedianPruner(n_warmup_steps=10))
    st.optimize(obj, n_trials=20)
    print(f"Optuna+pruning best AUC={st.best_value:.4f}, pruned="
          f"{len([1 for tr in st.trials if tr.state.name=='PRUNED'])}/20")
except ImportError:
    clf = xgb.XGBClassifier(n_estimators=500, learning_rate=0.1, max_depth=4,
                            eval_metric="auc", early_stopping_rounds=20,
                            random_state=42, n_jobs=1)
    clf.fit(Xt, yt, eval_set=[(Xv, yv)], verbose=False)
    auc = roc_auc_score(yv, clf.predict_proba(Xv)[:, 1])
    print("optuna no instalado -> ilustramos el pruning con early stopping de XGBoost")
    print(f"mejor iteracion={clf.best_iteration} de 500 (el resto se 'poda') | AUC val={auc:.4f}")
    assert clf.best_iteration < 500, "early stopping debe cortar antes de agotar los 500 rounds"

**Ej. 4 — Persistencia del study.** `create_study(study_name, storage='sqlite:///opt.db', load_if_exists=True)` permite re-correr y **agregar** trials. Fallback: simulamos la reanudación guardando/recargando los resultados en un archivo temporal.

In [ ]:

import os, json, tempfile
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

db = os.path.join(tempfile.gettempdir(), "opt_clase058.db")
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    def obj(t):
        m = RandomForestClassifier(n_estimators=t.suggest_int("n_estimators", 50, 200),
                                   max_depth=t.suggest_int("max_depth", 2, 10),
                                   random_state=42, n_jobs=1)
        return cross_val_score(m, X, y, cv=cv, scoring="roc_auc").mean()
    kw = dict(study_name="exp1", storage=f"sqlite:///{db}", load_if_exists=True, direction="maximize")
    optuna.create_study(**kw).optimize(obj, n_trials=5)   # primera corrida
    study = optuna.create_study(**kw)                     # reanuda
    study.optimize(obj, n_trials=5)                       # agrega 5 mas
    print(f"study reanudado: {len(study.trials)} trials totales (5+5)")
    assert len(study.trials) >= 10
except ImportError:
    path = os.path.join(tempfile.gettempdir(), "trials_clase058.json")
    rng = np.random.default_rng(0)
    def run(nt):
        prev = json.load(open(path)) if os.path.exists(path) else []
        for _ in range(nt):
            n = int(rng.integers(50, 200))
            m = RandomForestClassifier(n_estimators=n, random_state=42, n_jobs=1)
            prev.append({"n_estimators": n,
                         "auc": cross_val_score(m, X, y, cv=cv, scoring="roc_auc").mean()})
        json.dump(prev, open(path, "w")); return prev
    run(5); total = run(5)   # dos corridas: la segunda reanuda desde el archivo
    print("optuna no instalado -> reanudacion simulada con archivo JSON")
    print(f"trials acumulados tras 5+5: {len(total)}")
    assert len(total) >= 10
    os.remove(path)

**Ej. 5 — Multi-objective (Pareto).** Maximizar AUC *y* minimizar tiempo de inferencia. El frente de Pareto son las soluciones no dominadas (ninguna otra es mejor en ambos ejes a la vez). Fallback: lo computamos a mano sobre varios modelos.

In [ ]:

import time
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

Xt, Xv, yt, yv = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

def evaluar(n_estimators):
    m = RandomForestClassifier(n_estimators=n_estimators, random_state=42, n_jobs=1).fit(Xt, yt)
    t0 = time.perf_counter(); m.predict_proba(Xv); t = time.perf_counter() - t0
    return roc_auc_score(yv, m.predict_proba(Xv)[:, 1]), t

puntos = [(n, *evaluar(n)) for n in (10, 25, 50, 100, 200, 400)]

def domina(a, b):  # a domina a b si AUC>= y tiempo<= y estricto en alguno
    return a[1] >= b[1] and a[2] <= b[2] and (a[1] > b[1] or a[2] < b[2])
pareto = [p for p in puntos if not any(domina(q, p) for q in puntos if q is not p)]

print("n_estimators |  AUC   | t_infer (s)")
for n, auc, t in puntos:
    flag = "  <- Pareto" if (n, auc, t) in pareto else ""
    print(f"{n:11d} | {auc:.4f} | {t:.5f}{flag}")
print(f"\nfrente de Pareto: {sorted(p[0] for p in pareto)} (mas AUC cuesta mas tiempo)")
assert len(pareto) >= 1